# PrimeInsurance - Claim Risk & Anomaly Engine
**Catalog:** primeins | **Source:** primeins.silver.silver_claims | **Output:** primeins.gold.claim_anomaly_*

In [0]:
# Databricks Notebook Source
# MAGIC %md
# MAGIC 

# COMMAND ----------

%pip install -U mlflow pyrepo-mcda openai

dbutils.library.restartPython()


## Imports

In [0]:
from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
from openai import OpenAI
from pyrepo_mcda import weighting_methods as mcda_weights
import mlflow
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

matplotlib.use("Agg")

# Suppress the Pydantic serialization UserWarning emitted by MLflow autologging
warnings.filterwarnings(
    "ignore",
    message="Pydantic serializer warnings.*",
    category=UserWarning,
    module="pydantic",
)
mlflow.openai.autolog()


## Pipeline-level constants

In [0]:
RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")
CATALOG  = "dbx_hackathon_prime_insurance"
SCHEMA_S = f"{CATALOG}.silver"
SCHEMA_G = f"{CATALOG}.gold"


## LLM client - Databricks Foundation Model endpoint.

In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")

client   = OpenAI(
    api_key  = DATABRICKS_TOKEN,
    base_url = "https://842446108592895.ai-gateway.cloud.databricks.com/mlflow/v1/"
)
MODEL_ID = "databricks-gpt-oss-20b"

print(f"Model  : {MODEL_ID}")
print(f"Run ID : {RUN_ID}")


## ClaimsEnricher 
### Loads and enriches silver claims for downstream scoring

In [0]:
class ClaimsEnricher:
    """Loads and enriches silver claims for downstream scoring.

    Responsibilities:
    - Read active rows from primeins.silver.silver_claims (SCD-2 aware).
    - LEFT JOIN dim_policy and dim_customer to add customer_id, policy_csl,
      and customer_region. Stubs out missing columns rather than raising so
      the pipeline degrades gracefully when dimensions are not yet populated.
    - Cast and normalise numeric columns (try_cast - never hard-fails on bad data).
    - Derive total_claim_amount, claim_processing_days, and canonical component
      aliases (injury_claim, vehicle_claim, property_claim).
    - Compute per-customer claim count used as context in AI investigation briefs.
    """

    def __init__(self, schema_silver: str, schema_gold: str):
        self.schema_s = schema_silver
        self.schema_g = schema_gold

    # -- public entry point ---------------------------------------------------

    def load(self) -> tuple:
        """Load, enrich, and return the ready-to-score claims DataFrame.

        Returns:
            (claims, total_claims) - enriched DataFrame and its row count.
        """
        df = self._read_current(f"{self.schema_s}.silver_claims")
        print(f"silver_claims loaded: {df.count():,} rows")

        df = self._join_policy(df)
        df = self._join_customer(df)
        df = self._cast_numerics(df)
        df = self._derive_amounts(df)
        df = self._derive_processing_days(df)
        df = self._add_claim_count(df)

        total = df.count()
        print(f"\nClaims ready for scoring : {total:,}")
        print(
            f"Amount  max={df.agg(F.max('total_claim_amount')).collect()[0][0]:,.0f}"
            f"  mean={df.agg(F.mean('total_claim_amount')).collect()[0][0]:,.0f}"
        )
        return df, total

    # -- private helpers ------------------------------------------------------

    @staticmethod
    def _read_current(table: str) -> DataFrame:
        """Return only active rows, stripping SCD-2 temporal columns if present."""
        df = spark.table(table)
        if "__END_AT" in df.columns:
            df = df.filter(F.col("__END_AT").isNull()).drop("__START_AT", "__END_AT")
        return df

    def _join_policy(self, df: DataFrame) -> DataFrame:
        """LEFT JOIN dim_policy to add customer_id and policy_csl."""
        try:
            dim = self._read_current(f"{self.schema_g}.dim_policy")
            needed = [c for c in ["policy_number", "customer_id", "policy_csl"] if c in dim.columns]
            if "policy_number" not in dim.columns:
                raise ValueError("policy_number missing from dim_policy")
            slim = dim.select(*needed).withColumnRenamed("policy_number", "policyid")
            df = df.join(slim, on="policyid", how="left")
            print(f"OK dim_policy joined - added: {[c for c in needed if c != 'policy_number']}")
        except Exception as e:
            print(f"WARNING dim_policy unavailable - skipping: {e}")
            for col in ["customer_id", "policy_csl"]:
                if col not in df.columns:
                    df = df.withColumn(col, F.lit(None).cast(T.StringType()))
        return df

    def _join_customer(self, df: DataFrame) -> DataFrame:
        """LEFT JOIN dim_customer to add customer_region."""
        try:
            dim = self._read_current(f"{self.schema_g}.dim_customer")
            if "customer_id" not in dim.columns or "region" not in dim.columns:
                raise ValueError("customer_id or region missing from dim_customer")
            slim = dim.select("customer_id", F.col("region").alias("customer_region"))
            df = df.join(slim, on="customer_id", how="left")
            print("OK dim_customer joined - customer_region added")
        except Exception as e:
            print(f"WARNING dim_customer unavailable - skipping: {e}")
            if "customer_region" not in df.columns:
                df = df.withColumn("customer_region", F.lit(None).cast(T.StringType()))
        return df

    def _cast_numerics(self, df: DataFrame) -> DataFrame:
        """Cast amount and count columns to double via try_cast (never hard-fails)."""
        for col in ["injury", "vehicle", "property"]:
            if col in df.columns:
                df = df.withColumn(col, F.expr(f"try_cast(`{col}` as double)"))
        for col in ["witnesses", "number_of_vehicles_involved", "bodily_injuries"]:
            if col in df.columns:
                df = df.withColumn(col, F.expr(f"try_cast(`{col}` as double)"))
        return df

    def _derive_amounts(self, df: DataFrame) -> DataFrame:
        """Derive total_claim_amount and canonical component aliases.

        coalesce to 0 so a NULL component does not null-out the total.
        """
        df = df.withColumn(
            "total_claim_amount",
            F.coalesce(F.col("injury"), F.lit(0.0)) +
            F.coalesce(F.col("vehicle"), F.lit(0.0)) +
            F.coalesce(F.col("property"), F.lit(0.0))
        )
        return (
            df.withColumn("injury_claim",   F.col("injury"))
              .withColumn("vehicle_claim",  F.col("vehicle"))
              .withColumn("property_claim", F.col("property"))
        )

    def _derive_processing_days(self, df: DataFrame) -> DataFrame:
        """Derive claim_processing_days from log/processed dates if not already present."""
        date_cols = ["claim_logged_on", "claim_processed_on", "incident_date"]
        if "claim_processing_days" not in df.columns:
            if "claim_logged_on" in df.columns and "claim_processed_on" in df.columns:
                for dc in date_cols:
                    if dc in df.columns:
                        df = df.withColumn(dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')"))
                df = df.withColumn(
                    "claim_processing_days",
                    F.datediff(F.col("claim_processed_on"), F.col("claim_logged_on")).cast(T.DoubleType())
                )
            else:
                df = df.withColumn("claim_processing_days", F.lit(None).cast(T.DoubleType()))
        else:
            for dc in date_cols:
                if dc in df.columns:
                    df = df.withColumn(dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')"))
            df = df.withColumn(
                "claim_processing_days", F.expr("try_cast(claim_processing_days as double)")
            )
        return df

    def _add_claim_count(self, df: DataFrame) -> DataFrame:
        """Add per-customer (or per-policy) claim count used as brief context."""
        if "customer_id" in df.columns and df.filter(F.col("customer_id").isNotNull()).count() > 0:
            key = "customer_id"
        else:
            key = "policyid"
            print(f"WARNING: customer_id unavailable - using {key} for claim count")
        if "customer_claim_count" in df.columns:
            df = df.drop("customer_claim_count")
        counts = df.groupBy(key).agg(F.count("claimid").alias("customer_claim_count"))
        df = df.join(F.broadcast(counts), on=key, how="left")
        df = df.withColumn("customer_claim_count", F.coalesce(F.col("customer_claim_count"), F.lit(1)))
        high_freq = df.filter(F.col("customer_claim_count") > F.lit(1)).select(key).distinct().count()
        print(f"Claim count keyed on : {key}")
        print(f"High-frequency (>1)  : {high_freq}")
        return df


## AnomalyScorer
### Computes per-claim anomaly scores using eight statistical rule
#### Scoring rules:
        R1  Amount Z-Score       - total amount vs population mean
        R2  Severity Mismatch    - amount unusually high for the severity tier
        R3  Injury/Vehicle Ratio - injury payout disproportionate to vehicle damage
        R4  Staged Accident      - multi-vehicle + no witnesses + no police
        R5  Documentation Gap    - missing property_damage / police report / authorities
        R6  Theft with Injury    - vehicle theft logically cannot produce bodily injuries
        R7  Ghost Injury         - injury payout with zero reported bodily injuries
        R8  Injury Per Person    - disproportionate injury cost per reported bodily injury

#### Weighting strategy
- CRITIC method (objective):
        Assigns higher weight to rules with HIGH variance and LOW inter-rule
        correlation. Weights are data-driven and reproducible under audit.

- Business weights (comparative):
        Domain-expert-assigned weights.

#### Priority tiers (score out of 100):
- HIGH   >= HIGH_THRESHOLD (65)
- MEDIUM >= LOWER_THRESHOLD (40) and < HIGH_THRESHOLD
- LOW     < LOWER_THRESHOLD



In [0]:
class AnomalyScorer:
    """Computes per-claim anomaly scores using eight statistical rules """

    # Thresholds - gate tier assignment in _apply_critic_weights and _apply_business_weights
    LOWER_THRESHOLD = 40
    HIGH_THRESHOLD  = 65

    # Rule column names - order must be preserved for CRITIC matrix alignment
    INDICATOR_COLS = [
        "r1_amount_zscore",
        "r2_severity_mismatch",
        "r3_injury_ratio",
        "r4_staged_accident",
        "r5_doc_gap",
        "r6_theft_with_injury",
        "r7_ghost_injury",
        "r8_injury_per_person",
    ]

    # Business weights - domain-expert assignment for comparative scoring
    # R1 18%  strong signal, penalised for R2 overlap
    # R2 16%  strong but correlated with R1
    # R3 14%  reliable fraud pattern
    # R4 10%  high specificity, lower coverage
    # R5  7%  amplifier, rarely fires alone
    # R6  5%  low statistical signal, high domain validity
    # R7 13%  r=0.48, clear manufactured claim pattern
    # R8 17%  r=0.72, strongest rule
    BUSINESS_WEIGHTS = {
        "r1_amount_zscore":     0.18,
        "r2_severity_mismatch": 0.16,
        "r3_injury_ratio":      0.14,
        "r4_staged_accident":   0.10,
        "r5_doc_gap":           0.07,
        "r6_theft_with_injury": 0.05,
        "r7_ghost_injury":      0.13,
        "r8_injury_per_person": 0.17,
    }

    def __init__(self):
        assert abs(sum(self.BUSINESS_WEIGHTS.values()) - 1.0) < 1e-9, \
            "BUSINESS_WEIGHTS must sum to 1.0"
        self.critic_weights = None  # populated by _apply_critic_weights
        self.corr_matrix    = None

    def score(self, df: DataFrame) -> DataFrame:
        """Run the full scoring pipeline in order.

        Calls population stats, all eight rules, both weighting methods,
        and triggered_rules assembly. Each step is defined in its own cell below.

        Args:
            df: Enriched claims DataFrame from ClaimsEnricher.

        Returns:
            DataFrame with all rule scores, composite scores, tiers,
            and triggered_rules column added.
        """
        df = self._compute_population_stats(df)
        df = self._r1_amount_zscore(df)
        df = self._r2_severity_mismatch(df)
        df = self._r3_injury_ratio(df)
        df = self._r4_staged_accident(df)
        df = self._r5_doc_gap(df)
        df = self._r6_theft_with_injury(df)
        df = self._r7_ghost_injury(df)
        df = self._r8_injury_per_person(df)
        df = self._apply_critic_weights(df)
        df = self._apply_business_weights(df)
        df = self._assign_triggered_rules(df)
        return df


### Population statistics - collected once for use across all rules.

In [0]:

def _compute_population_stats(self, df: DataFrame) -> DataFrame:
    """Collect global and per-severity amount statistics.

    Stores results as instance attributes (_amt_mean, _amt_std, _sev_dict)
    so all downstream rule methods can reference them without recomputing.
    """
    row = df.agg(
        F.mean("total_claim_amount").alias("amt_mean"),
        F.stddev_pop("total_claim_amount").alias("amt_std"),
    ).collect()[0]
    self._amt_mean = float(row["amt_mean"] or 0)
    self._amt_std  = float(row["amt_std"]  or 1)

    sev_rows = (
        df.filter(F.col("incident_severity").isNotNull())
          .groupBy("incident_severity")
          .agg(
              F.mean("total_claim_amount").alias("sev_mean"),
              F.stddev_pop("total_claim_amount").alias("sev_std"),
          )
          .collect()
    )
    self._sev_dict = {
        r["incident_severity"]: (float(r["sev_mean"] or 0), float(r["sev_std"] or 1))
        for r in sev_rows
    }
    print(f"Population: amt_mean={self._amt_mean:,.0f}  amt_std={self._amt_std:,.0f}")
    print(f"Severity map: {self._sev_dict}")
    return df


AnomalyScorer._compute_population_stats = _compute_population_stats


#### R1 - Amount Z-Score
- Detects claims with inflated total amounts relative to the population.
- Z-score capped at 2 std-devs and normalised to [0, 1] so all rule scores occupy the same range before weighting.

In [0]:
def _r1_amount_zscore(self, df: DataFrame) -> DataFrame:
    df = df.withColumn(
        "_r1_raw",
        F.when(
            F.col("total_claim_amount").isNotNull(),
            (F.col("total_claim_amount") - F.lit(self._amt_mean)) / F.lit(self._amt_std)
        ).otherwise(F.lit(0.0))
    )
    return df.withColumn(
        "r1_amount_zscore",
        F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r1_raw") / F.lit(2.0)))
    ).drop("_r1_raw")


AnomalyScorer._r1_amount_zscore = _r1_amount_zscore


#### R2 - Severity Mismatch
- Detects claims priced abnormally high within their severity peer group.
- A "Trivial" claim priced like "Major" is a strong indicator of intentional severity mislabeling to inflate the payout.


In [0]:
def _r2_severity_mismatch(self, df: DataFrame) -> DataFrame:
    mean_expr = F.lit(self._amt_mean)
    std_expr  = F.lit(self._amt_std)
    for sev, (m, s) in self._sev_dict.items():
        mean_expr = F.when(F.col("incident_severity") == sev, F.lit(m)).otherwise(mean_expr)
        std_expr  = F.when(F.col("incident_severity") == sev, F.lit(max(s, 1.0))).otherwise(std_expr)
    df = df.withColumn(
        "_r2_z",
        F.when(
            F.col("total_claim_amount").isNotNull(),
            (F.col("total_claim_amount") - mean_expr) / std_expr
        ).otherwise(F.lit(0.0))
    )
    return df.withColumn(
        "r2_severity_mismatch",
        F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r2_z") / F.lit(2.0)))
    ).drop("_r2_z")


AnomalyScorer._r2_severity_mismatch = _r2_severity_mismatch



#### R3 - Injury/Vehicle Ratio Anomaly
- Injury claims are harder to verify than vehicle repair costs, making them the preferred inflation target in staged accident fraud.
- +1 denominator prevents division-by-zero on zero-vehicle-damage claims.

In [0]:

def _r3_injury_ratio(self, df: DataFrame) -> DataFrame:
    row = (
        df.withColumn("_ratio", F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0)))
          .agg(
              F.mean("_ratio").alias("ratio_mean"),
              F.stddev_pop("_ratio").alias("ratio_std"),
          )
          .collect()[0]
    )
    ratio_mean = float(row["ratio_mean"] or 0)
    ratio_std  = float(row["ratio_std"]  or 1)
    print(f"R3 ratio: mean={ratio_mean:.3f}  std={ratio_std:.3f}")

    df = df.withColumn(
        "_r3_z",
        F.when(
            F.col("injury_claim").isNotNull() & F.col("vehicle_claim").isNotNull(),
            ((F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0))) - F.lit(ratio_mean))
            / F.lit(max(ratio_std, 0.001))
        ).otherwise(F.lit(0.0))
    )
    return df.withColumn(
        "r3_injury_ratio",
        F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r3_z") / F.lit(2.0)))
    ).drop("_r3_z")


AnomalyScorer._r3_injury_ratio = _r3_injury_ratio


#### R4 - Staged Accident
- Multi-vehicle collisions almost always involve witnesses and law enforcement.
- The simultaneous absence of both is a classic staged-accident pattern.
- Score = (vehicles > 1  +  witnesses == 0  +  no police report) / 3
- Produces one of four values: {0.0, 0.33, 0.67, 1.0}

In [0]:
def _r4_staged_accident(self, df: DataFrame) -> DataFrame:
    has_nv = "number_of_vehicles_involved" in df.columns
    has_wi = "witnesses"               in df.columns
    has_pr = "police_report_available" in df.columns

    c1 = (F.col("number_of_vehicles_involved") > F.lit(1)).cast(T.DoubleType()) if has_nv else F.lit(0.0)
    c2 = (F.col("witnesses") == F.lit(0)).cast(T.DoubleType())                   if has_wi else F.lit(0.0)
    c3 = (F.upper(F.col("police_report_available")) != F.lit("YES")).cast(T.DoubleType()) if has_pr else F.lit(0.0)

    return df.withColumn(
        "r4_staged_accident",
        F.round(
            (F.coalesce(c1, F.lit(0.0)) + F.coalesce(c2, F.lit(0.0)) + F.coalesce(c3, F.lit(0.0)))
            / F.lit(3.0), 4
        )
    )


AnomalyScorer._r4_staged_accident = _r4_staged_accident


#### R5 - Documentation Gap
- Missing documentation is a hallmark of manufactured claims - genuine claimants have every incentive to submit supporting evidence.
- Score = (property_damage NULL  +  no police report  +  authorities == None) / 2
- Divided by 2 intentionally: positions this as an amplifier, not a primary signal.


In [0]:
def _r5_doc_gap(self, df: DataFrame) -> DataFrame:
    has_pr   = "police_report_available" in df.columns
    has_pd   = "property_damage"         in df.columns
    has_auth = "authorities_contacted"   in df.columns

    s1 = F.col("property_damage").isNull().cast(T.DoubleType()) if has_pd else F.lit(0.0)
    s2 = (
        F.col("police_report_available").isNull() |
        (F.upper(F.col("police_report_available")) != F.lit("YES"))
    ).cast(T.DoubleType()) if has_pr else F.lit(0.0)
    s3 = (F.lower(F.col("authorities_contacted")) == F.lit("none")).cast(T.DoubleType()) if has_auth else F.lit(0.0)

    return df.withColumn(
        "r5_doc_gap",
        F.round(
            (F.coalesce(s1, F.lit(0.0)) + F.coalesce(s2, F.lit(0.0)) + F.coalesce(s3, F.lit(0.0)))
            / F.lit(2.0), 4
        )
    )


AnomalyScorer._r5_doc_gap = _r5_doc_gap


#### R6 - Vehicle Theft with Bodily Injuries
- Logical impossibility: a vehicle theft cannot produce bodily injuries.
- Statistically weak (r ~ 0.02) but domain-defensible; included at low weight.
- Score: binary 1.0 if both conditions are present, 0.0 otherwise.

In [0]:

def _r6_theft_with_injury(self, df: DataFrame) -> DataFrame:
    if "incident_type" in df.columns and "bodily_injuries" in df.columns:
        df = df.withColumn(
            "r6_theft_with_injury",
            F.when(
                (F.lower(F.col("incident_type")) == F.lit("vehicle theft")) &
                (F.col("bodily_injuries") > F.lit(0)),
                F.lit(1.0)
            ).otherwise(F.lit(0.0))
        )
    else:
        df = df.withColumn("r6_theft_with_injury", F.lit(0.0))
    print(f"R6 - theft with injury: {df.filter(F.col('r6_theft_with_injury') == 1.0).count()} flagged")
    return df


AnomalyScorer._r6_theft_with_injury = _r6_theft_with_injury


#### R7 - Ghost Injury 

- Bodily_injuries = 0 combined with a non-zero injury_claim is a logical contradiction and a strong indicator of a manufactured claim component.
- Within-ghost-population z-score, clipped to [0, 1].
- Non-ghost claims always score 0.0.

In [0]:
def _r7_ghost_injury(self, df: DataFrame) -> DataFrame:
    if "bodily_injuries" not in df.columns or "injury_claim" not in df.columns:
        return df.withColumn("r7_ghost_injury", F.lit(0.0))

    row = (
        df.filter(
            (F.col("bodily_injuries") == F.lit(0)) &
            F.col("injury_claim").isNotNull() &
            (F.col("injury_claim") > F.lit(0))
        )
        .agg(
            F.mean("injury_claim").alias("ghost_mean"),
            F.stddev_pop("injury_claim").alias("ghost_std"),
        )
        .collect()[0]
    )
    ghost_mean = float(row["ghost_mean"] or 0)
    ghost_std  = float(row["ghost_std"]  or 1)
    print(f"R7 ghost stats: mean=${ghost_mean:,.0f}  std=${ghost_std:,.0f}")

    df = df.withColumn(
        "r7_ghost_injury",
        F.when(
            (F.col("bodily_injuries") == F.lit(0)) &
            F.col("injury_claim").isNotNull() &
            (F.col("injury_claim") > F.lit(0)),
            F.least(
                F.lit(1.0),
                F.greatest(
                    F.lit(0.0),
                    (F.col("injury_claim") - F.lit(ghost_mean)) /
                    F.lit(max(ghost_std, 1.0)) / F.lit(2.0)
                )
            )
        ).otherwise(F.lit(0.0))
    )
    print(f"R7 - ghost injury (bodily=0, injury>0): {df.filter(F.col('r7_ghost_injury') > 0).count()}")
    return df


AnomalyScorer._r7_ghost_injury = _r7_ghost_injury



#### R8 - Injury Per Person  (r = 0.72, p < 0.0001 - strongest rule)
- Normalises injury_claim by the number of reported bodily injuries
- A $50,000 claim with 1 bodily injury is far more suspicious than the same total spread across 4 injuries
- Per-person normalisation catches disproportionate per-head costs that R3 (ratio to vehicle damage) misses entirely

In [0]:

def _r8_injury_per_person(self, df: DataFrame) -> DataFrame:
    if "bodily_injuries" not in df.columns or "injury_claim" not in df.columns:
        return df.withColumn("r8_injury_per_person", F.lit(0.0))

    df = df.withColumn(
        "_ipp",
        F.when(F.col("injury_claim").isNull(), F.lit(0.0))
         .when(F.col("bodily_injuries") > F.lit(0),
               F.col("injury_claim") / F.col("bodily_injuries"))
         .otherwise(F.col("injury_claim"))  # ghost-injury amplifier
    )
    row = df.agg(
        F.mean("_ipp").alias("ipp_mean"),
        F.stddev_pop("_ipp").alias("ipp_std"),
    ).collect()[0]
    ipp_mean = float(row["ipp_mean"] or 0)
    ipp_std  = float(row["ipp_std"]  or 1)
    print(f"R8 injury-per-person: mean=${ipp_mean:,.0f}  std=${ipp_std:,.0f}")

    return df.withColumn(
        "r8_injury_per_person",
        F.least(
            F.lit(1.0),
            F.greatest(
                F.lit(0.0),
                (F.col("_ipp") - F.lit(ipp_mean)) / F.lit(max(ipp_std, 1.0)) / F.lit(2.0)
            )
        )
    ).drop("_ipp")


AnomalyScorer._r8_injury_per_person = _r8_injury_per_person


### Rules Not Implemented — Data Limitations

Two candidate fraud rules were evaluated and excluded from the scoring engine due to data quality constraints identified during the Silver pipeline stage.

**Delayed Claim Processing**
Claims with unusually long processing times (> 30 days) are a recognised fraud and backlog indicator. However, `claim_logged_on` and `claim_processed_on` in the source data contain only Excel time-serial fragments (e.g. `"27:00.0"`) — no recoverable date can be derived from these fields. Affected records were quarantined at the Silver layer, making `claim_processing_days` = `NaN` across the dataset and this rule unscoreable.
**Resolution:** once upstream data capture is fixed to record full ISO dates, this rule can be added as a direct z-score on `claim_processing_days` with a 30-day backlog threshold.

**Repeat Claims per Customer**
Customers filing multiple claims within a short window are a well-established fraud pattern. However, every customer in this dataset has exactly **1 claim** — confirmed by `High-frequency (>1): 0` in the enrichment output. With no variance in claim frequency across the population, a repeat-claim rule would score every record identically and contribute zero discriminatory power.
**Resolution:** this rule is structurally ready — `customer_claim_count` is already derived and carried through the pipeline — and will activate automatically once the dataset contains customers with multiple claims.


#### CRITIC Weighting - objective, data-driven weight derivation.

- CRITIC (CRiteria Importance Through Intercriteria Correlation) assigns higher weight to rules with 
    
    (a) HIGH variance  - strong discriminatory power across the population
    
    (b) LOW correlation - unique, non-redundant signal (penalises collinear rules)

- Weights are fully reproducible and update automatically as new claims data accumulates

In [0]:
def _apply_critic_weights(self, df: DataFrame) -> DataFrame:
    """Derive CRITIC weights, compute anomaly_score, assign priority_tier."""
    matrix = df.select(*self.INDICATOR_COLS).fillna(0).toPandas().to_numpy()
    stds             = matrix.std(axis=0)
    self.corr_matrix = np.corrcoef(matrix.T)
    conflict         = np.array([
        sum(1 - abs(self.corr_matrix[i][j]) for j in range(len(self.INDICATOR_COLS)) if j != i)
        for i in range(len(self.INDICATOR_COLS))
    ])
    self.critic_weights = mcda_weights.critic_weighting(matrix)
    W = self.critic_weights

    print("\n" + "=" * 60)
    print("CRITIC - FINAL WEIGHTS")
    print("=" * 60)
    for col, w, s, c in zip(self.INDICATOR_COLS, W, stds, conflict):
        bar = "X" * int(w * 40)
        print(f"{col:<28} w={w:.4f}  std={s:.4f}  conflict={c:.4f}  {bar}")
    print(f"\nWeights sum: {W.sum():.6f}")

    print("\nCorrelation matrix (! = |r| > 0.90):")
    labels = ["R1","R2","R3","R4","R5","R6","R7","R8"]
    print(f"{''  :>6}" + "".join(f"{l:>8}" for l in labels))
    for i, lbl in enumerate(labels):
        row_str = f"{lbl:>6}"
        for j in range(len(labels)):
            val    = self.corr_matrix[i][j]
            marker = "!" if i != j and abs(val) > 0.9 else " "
            row_str += f"{val:>7.3f}{marker}"
        print(row_str)

    w1,w2,w3,w4,w5,w6,w7,w8 = [float(x) for x in W]
    df = df.withColumn(
        "anomaly_score",
        F.round(F.lit(100.0) * (
            F.lit(w1) * F.col("r1_amount_zscore")    +
            F.lit(w2) * F.col("r2_severity_mismatch") +
            F.lit(w3) * F.col("r3_injury_ratio")      +
            F.lit(w4) * F.col("r4_staged_accident")   +
            F.lit(w5) * F.col("r5_doc_gap")           +
            F.lit(w6) * F.col("r6_theft_with_injury") +
            F.lit(w7) * F.col("r7_ghost_injury")      +
            F.lit(w8) * F.col("r8_injury_per_person")
        ), 2)
    )
    return df.withColumn(
        "priority_tier",
        F.when(F.col("anomaly_score") >= F.lit(self.HIGH_THRESHOLD),   F.lit("HIGH"))
         .when(F.col("anomaly_score") >= F.lit(self.LOWER_THRESHOLD),  F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )


AnomalyScorer._apply_critic_weights = _apply_critic_weights



#### Business Weights - domain-expert assignment for comparative scoring.
- Runs in parallel with CRITIC weights to produce anomaly_score_bw and
priority_tier_bw.

In [0]:

def _apply_business_weights(self, df: DataFrame) -> DataFrame:
    """Compute anomaly_score_bw and priority_tier_bw using BUSINESS_WEIGHTS."""
    BW = self.BUSINESS_WEIGHTS
    bw1,bw2,bw3,bw4,bw5,bw6,bw7,bw8 = [BW[c] for c in self.INDICATOR_COLS]
    df = df.withColumn(
        "anomaly_score_bw",
        F.round(F.lit(100.0) * (
            F.lit(bw1) * F.col("r1_amount_zscore")    +
            F.lit(bw2) * F.col("r2_severity_mismatch") +
            F.lit(bw3) * F.col("r3_injury_ratio")      +
            F.lit(bw4) * F.col("r4_staged_accident")   +
            F.lit(bw5) * F.col("r5_doc_gap")           +
            F.lit(bw6) * F.col("r6_theft_with_injury") +
            F.lit(bw7) * F.col("r7_ghost_injury")      +
            F.lit(bw8) * F.col("r8_injury_per_person")
        ), 2)
    )
    return df.withColumn(
        "priority_tier_bw",
        F.when(F.col("anomaly_score_bw") >= F.lit(self.HIGH_THRESHOLD),  F.lit("HIGH"))
         .when(F.col("anomaly_score_bw") >= F.lit(self.LOWER_THRESHOLD), F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )


AnomalyScorer._apply_business_weights = _apply_business_weights


#### Triggered Rules 
- Human-readable summary column for investigator display.
- Appends a comma-separated string of rule labels for each claim.

In [0]:

# Thresholds here are display thresholds (when to show a rule as "fired"),
# not the scoring thresholds - kept slightly lower to surface borderline signals.

def _assign_triggered_rules(self, df: DataFrame) -> DataFrame:
    """Build triggered_rules string from per-rule score columns."""
    return df.withColumn(
        "triggered_rules",
        F.concat_ws(", ",
            F.when(F.col("r1_amount_zscore")    >= 0.4, F.lit("R1:HighAmount")).otherwise(F.lit(None)),
            F.when(F.col("r2_severity_mismatch") >= 0.4, F.lit("R2:SeverityMismatch")).otherwise(F.lit(None)),
            F.when(F.col("r3_injury_ratio")      >= 0.4, F.lit("R3:InjuryRatio")).otherwise(F.lit(None)),
            F.when(F.col("r4_staged_accident")   >= 0.4, F.lit("R4:StagedAccident")).otherwise(F.lit(None)),
            F.when(F.col("r5_doc_gap")           >= 0.3, F.lit("R5:DocGap")).otherwise(F.lit(None)),
            F.when(F.col("r6_theft_with_injury") == 1.0, F.lit("R6:TheftWithInjury")).otherwise(F.lit(None)),
            F.when(F.col("r7_ghost_injury")      >= 0.4, F.lit("R7:GhostInjury")).otherwise(F.lit(None)),
            F.when(F.col("r8_injury_per_person") >= 0.4, F.lit("R8:InjuryPerPerson")).otherwise(F.lit(None)),
        )
    )


AnomalyScorer._assign_triggered_rules = _assign_triggered_rules


#### AnomalyReporter
- Generates AI investigation briefs and writes all output Gold tables

In [0]:
# COMMAND ----------


class AnomalyReporter:
    """Generates AI investigation briefs and writes all output Gold tables.

    Responsibilities:
    - Build structured prompts from scored claim rows.
    - Call the LLM with retry logic; store error sentinels so no record is lost.
    - Render CRITIC vs Business Weights comparison (Venn + bar chart).
    - Write claim_anomaly_scores, claim_anomaly_explanations,
      and claim_anomaly_weights to primeins.gold.
    """

    # Columns forwarded to the LLM prompt and the explanations output table
    BRIEF_COLS = [
        "claimid", "customer_id", "policyid",
        "incident_date", "incident_severity", "incident_type",
        "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
        "claim_processing_days",
        "property_damage", "police_report_available", "authorities_contacted",
        "number_of_vehicles_involved", "witnesses",
        "claim_rejected", "customer_claim_count",
        "anomaly_score", "priority_tier", "triggered_rules",
        "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
        "r4_staged_accident", "r5_doc_gap",
        "customer_region", "policy_csl",
    ]

    # Columns written to the scores table (all claims, no brief)
    SCORE_COLS = [
        "claimid", "customer_id", "policyid",
        "incident_date", "incident_severity", "incident_type",
        "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
        "claim_processing_days", "property_damage", "police_report_available",
        "authorities_contacted", "number_of_vehicles_involved", "witnesses",
        "claim_rejected", "customer_claim_count",
        "customer_region", "policy_csl",
        "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
        "r4_staged_accident", "r5_doc_gap",
        "anomaly_score", "priority_tier", "triggered_rules",
    ]

    def __init__(self, schema_gold: str, run_id: str, scorer: AnomalyScorer):
        self.schema_g = schema_gold
        self.run_id   = run_id
        self.scorer   = scorer  # access to critic_weights and corr_matrix for audit

    # -- public entry points --------------------------------------------------

    def plot_comparison(self, df: DataFrame) -> None:
        """Render a Venn + bar chart comparing CRITIC vs Business Weight flagging.

        Computes three agreement groups:
          - CRITIC only : flagged by CRITIC, not by business weights
          - Both         : flagged by both (high confidence)
          - BW only      : flagged by business weights, not by CRITIC
        """
        pdf = df.select(
            "claimid", "priority_tier", "priority_tier_bw",
        ).toPandas()

        critic_ids  = set(pdf[pdf["priority_tier"].isin(["HIGH","MEDIUM"])]["claimid"])
        bw_ids      = set(pdf[pdf["priority_tier_bw"].isin(["HIGH","MEDIUM"])]["claimid"])
        critic_only = critic_ids - bw_ids
        both        = critic_ids & bw_ids
        bw_only     = bw_ids - critic_ids

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        fig.patch.set_facecolor("#0f1117")

        ax = axes[0]
        ax.set_facecolor("#0f1117"); ax.set_xlim(0,10); ax.set_ylim(0,10)
        ax.set_aspect("equal"); ax.axis("off")
        ax.add_patch(Circle((3.6,5.0), 3.0, color="#4e9af1", alpha=0.35, zorder=2))
        ax.add_patch(Circle((6.4,5.0), 3.0, color="#f1914e", alpha=0.35, zorder=2))
        ax.text(2.2, 8.6, "CRITIC",           ha="center", fontsize=14, fontweight="bold", color="#4e9af1")
        ax.text(7.8, 8.6, "Business\nWeights", ha="center", fontsize=14, fontweight="bold", color="#f1914e")
        for x, val, lbl in [(2.1, len(critic_only), "CRITIC only"),
                             (5.0, len(both),        "both agree"),
                             (7.9, len(bw_only),     "BW only")]:
            ax.text(x, 5.2, str(val), ha="center", va="center", fontsize=44, fontweight="bold", color="white", zorder=3)
            ax.text(x, 4.1, lbl,      ha="center", fontsize=9,  color="#aaaaaa", zorder=3)
        ax.text(5.0, 1.0,
                f"CRITIC flagged {len(critic_ids)} total  |  BW flagged {len(bw_ids)} total  |  +{len(critic_only)} unique from CRITIC",
                ha="center", fontsize=9, color="white",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#1e2235", edgecolor="#4e9af1", linewidth=1.2))
        ax.set_title("Flagging Coverage Comparison", color="white", fontsize=13, fontweight="bold", pad=12)

        ax2 = axes[1]; ax2.set_facecolor("#0f1117"); ax2.axis("off")
        categories = ["CRITIC Total","Both Agree\n(High Confidence)","CRITIC Only\n(Unique Signals)","BW Only\n(Est. False Positives)"]
        values     = [len(critic_ids), len(both), len(critic_only), len(bw_only)]
        bar_colors = ["#4e9af1","#00e676","#7ec8f7","#f1914e"]
        max_val    = max(values) or 1
        for y, val, color, label in zip([3.2,2.4,1.6,0.8], values, bar_colors, categories):
            ax2.barh(y, max_val*1.1, 0.5, left=0, color="#1e2235", zorder=1)
            ax2.barh(y, val,         0.5, left=0, color=color, alpha=0.85, zorder=2)
            ax2.text(-0.5,             y, label,    ha="right", va="center", fontsize=9.5, color="#cccccc")
            ax2.text(val+max_val*0.02, y, str(val), ha="left",  va="center", fontsize=12,  fontweight="bold", color=color)
        ax2.text(max_val*0.55, 0.15,
                 f"CRITIC surfaces {len(critic_only)} additional signals\nthat BW misses - only {len(bw_only)} est. false positives.",
                 ha="center", va="center", fontsize=9, color="white",
                 bbox=dict(boxstyle="round,pad=0.5", facecolor="#1e2235", edgecolor="#00e676", linewidth=1.2))
        ax2.set_xlim(-max_val*0.6, max_val*1.2); ax2.set_ylim(0.2, 4.0)
        ax2.set_title("What Each Method Flags", color="white", fontsize=13, fontweight="bold", pad=12)

        fig.suptitle("CRITIC Outperforms Business Weights in Fraud Detection Coverage",
                     color="white", fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout(pad=2.5)
        display(fig)
        plt.close()

        corr = self.scorer.corr_matrix[0][1] if self.scorer.corr_matrix is not None else float("nan")
        print(f"\nCRITIC flagged {len(critic_ids)} vs {len(bw_ids)} by business weights.")
        print(f"Both agree on {len(both)} high-confidence cases - {len(critic_only)} additional signals from CRITIC.")
        print(f"Only {len(bw_only)} flagged by BW only - likely R1+R2 double-counting (r={corr:.3f}).")

    def generate_briefs(self, df: DataFrame) -> dict:
        """Generate AI investigation briefs for all flagged claims.

        Every claim with priority_tier HIGH or MEDIUM receives a brief.
        On LLM failure an '[LLM_ERROR ...]' sentinel is stored so no record
        is silently dropped.

        Args:
            df: Fully scored claims DataFrame.

        Returns:
            Dict mapping claimid (str) -> brief text.
        """
        available = [c for c in self.BRIEF_COLS if c in df.columns]
        flagged_pdf = (
            df.filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"]))
              .orderBy(F.desc("anomaly_score"))
              .select(*available)
              .fillna({"property_damage": "Unknown", "police_report_available": "Unknown"})
              .toPandas()
        )
        print(f"Generating AI briefs for {len(flagged_pdf)} flagged claims...\n")

        briefs = {}
        for i, (_, row) in enumerate(flagged_pdf.iterrows()):
            claim_id = str(row.get("claimid", f"row_{i}"))
            score    = row.get("anomaly_score", 0)
            tier     = row.get("priority_tier", "?")
            print(f"  [{i+1:>3}/{len(flagged_pdf)}]  claim={claim_id}  tier={tier}  score={score} ...", end=" ")
            brief            = self._call_llm(self._build_prompt(row.to_dict()))
            briefs[claim_id] = brief
            print(f"OK  '{brief[:60].replace(chr(10), ' ')}...'")

        errors = sum(1 for b in briefs.values() if b.startswith("[LLM_ERROR"))
        print(f"\nBriefs generated: {len(briefs)}  |  Errors: {errors}")
        return briefs

    def write_outputs(self, df: DataFrame, briefs: dict, total_claims: int) -> None:
        """Write all three Gold output tables.

        Tables written:
          claim_anomaly_scores       - every claim, scored and tiered
          claim_anomaly_explanations - flagged claims with AI briefs
          claim_anomaly_weights      - CRITIC weights per run for audit
        """
        flagged_count = df.filter(F.col("priority_tier").isin(["HIGH","MEDIUM"])).count()

        available_score_cols = [c for c in self.SCORE_COLS if c in df.columns]
        all_scored = (
            df.select(*available_score_cols)
              .withColumn("is_flagged",      F.col("priority_tier").isin(["HIGH","MEDIUM"]))
              .withColumn("_anomaly_run_id", F.lit(self.run_id))
              .withColumn("_gold_run_id",    F.lit(self.run_id))
              .withColumn("_gold_load_ts",   F.current_timestamp())
              .withColumn("_stage",          F.lit("claim_anomaly_scores"))
        )
        self._write_gold(all_scored, "claim_anomaly_scores",
                         f"ALL claims | run={self.run_id} | flagged={flagged_count:,}/{total_claims:,}")

        brief_rows = [(str(cid), txt) for cid, txt in briefs.items()]
        briefs_df  = spark.createDataFrame(brief_rows, schema=T.StructType([
            T.StructField("claimid",                T.StringType(), True),
            T.StructField("ai_investigation_brief", T.StringType(), True),
        ]))
        flagged_with_briefs = (
            all_scored.filter(F.col("is_flagged") == True)
                      .join(briefs_df, on="claimid", how="left")
                      .withColumn("_stage", F.lit("claim_anomaly_explanations"))
        )
        self._write_gold(flagged_with_briefs, "claim_anomaly_explanations",
                         f"FLAGGED | {flagged_count:,} briefs | model={MODEL_ID} | run={self.run_id}")

        W = self.scorer.critic_weights
        if W is not None:
            weight_rows   = [(col, float(W[i]), self.run_id) for i, col in enumerate(AnomalyScorer.INDICATOR_COLS)]
            weight_schema = T.StructType([
                T.StructField("rule_name",     T.StringType(), True),
                T.StructField("critic_weight", T.DoubleType(), True),
                T.StructField("_run_id",       T.StringType(), True),
            ])
            self._write_gold(
                spark.createDataFrame(weight_rows, schema=weight_schema),
                "claim_anomaly_weights",
                f"CRITIC weight audit | run={self.run_id}"
            )
            spark.table(f"{self.schema_g}.claim_anomaly_weights").show(truncate=False)

    # -- private helpers ------------------------------------------------------

    def _write_gold(self, df: DataFrame, table: str, comment: str = "") -> int:
        """Write DataFrame to a Gold Delta table with full overwrite.

        Idempotent - overwrites data and schema so upstream changes propagate
        without manual ALTER TABLE.
        """
        full = f"{self.schema_g}.{table}"
        (df.write
           .format("delta")
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(full))
        cnt = spark.table(full).count()
        print(f"OK  {full:<60} {cnt:>8,} rows  {comment}")
        return cnt

    def _call_llm(self, prompt: str, retries: int = 3, timeout: int = 30) -> str:
        """Call the LLM with a fixed inter-call delay and exponential backoff retry logic.

        A 1-second sleep before each call acts as the primary guard against hitting
        the workspace RPM ceiling. On 429 rate-limit errors, exponential backoff gives
        the endpoint token budget time to refill before retrying. Other transient
        failures (timeout, network) retry after a short fixed wait.
        Returns a structured '[LLM_ERROR ...]' sentinel after all attempts are
        exhausted so no record is silently lost.
        """
        import time

        time.sleep(1)   # inter-call delay — primary guard against RPM ceiling

        for attempt in range(1, retries + 2):
            try:
                response = client.chat.completions.create(
                    model      = MODEL_ID,
                    messages   = [{"role": "user", "content": prompt}],
                    max_tokens = 1024,
                    timeout    = timeout,
                )
                return self._extract_text(response)
            except Exception as e:
                err            = str(e)[:120]
                is_rate_limit  = "429" in err or "REQUEST_LIMIT_EXCEEDED" in err
                if attempt <= retries:
                    wait = (2 ** attempt) * 10 if is_rate_limit else 2
                    print(f"    WARNING attempt {attempt} failed ({err}) — retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    return f"[LLM_ERROR after {retries + 1} attempts: {err}]"

    @staticmethod
    def _extract_text(response) -> str:
        """Extract plain text from an LLM response.

        Handles two content formats:
          - Databricks Foundation Models: list of typed blocks
          - Standard OpenAI-compatible: plain string
        Skips reasoning and tool_use blocks; returns only type='text' content.
        """
        content = response.choices[0].message.content
        if isinstance(content, list):
            return " ".join(
                block.get("text", "")
                for block in content
                if isinstance(block, dict) and block.get("type") == "text"
            ).strip()
        return content.strip()

    def _build_prompt(self, row: dict) -> str:
        """Build a structured, data-grounded investigation brief prompt.

        Passes specific numeric indicator values so the LLM narrates the
        statistics rather than producing generic text. Every factual statement
        in the output is traceable to the anomaly score breakdown below.
        """
        score = float(row.get("anomaly_score", 0) or 0)
        r = lambda k: float(row.get(k, 0) or 0)
        return (
            "You are a senior fraud investigation analyst at PrimeInsurance.\n"
            f"A statistical anomaly engine has flagged claim {row.get('claimid', 'UNKNOWN')} for investigation.\n"
            "Your job is to write a concise, structured investigation brief that explains the findings.\n"
            "\n=== CLAIM DATA ===\n"
            f"Claim ID          : {row.get('claimid')}\n"
            f"Customer ID       : {row.get('customer_id')}\n"
            f"Policy ID         : {row.get('policyid')}\n"
            f"Incident Date     : {row.get('incident_date')}\n"
            f"Incident Severity : {row.get('incident_severity')}\n"
            f"Incident Type     : {row.get('incident_type')}\n"
            f"Total Claim       : {row.get('total_claim_amount', 0):,.2f}\n"
            f"  Vehicle Claim   : {row.get('vehicle_claim', 0):,.2f}\n"
            f"  Property Claim  : {row.get('property_claim', 0):,.2f}\n"
            f"  Injury Claim    : {row.get('injury_claim', 0):,.2f}\n"
            f"Processing Days   : {row.get('claim_processing_days')}\n"
            f"Vehicles Involved : {row.get('number_of_vehicles_involved')}\n"
            f"Witnesses         : {row.get('witnesses')}\n"
            f"Authorities       : {row.get('authorities_contacted')}\n"
            f"Property Damage   : {row.get('property_damage')}\n"
            f"Police Report     : {row.get('police_report_available')}\n"
            f"Claim Rejected    : {row.get('claim_rejected')}\n"
            f"Prior Claims      : {row.get('customer_claim_count')}\n"
            f"Region            : {row.get('customer_region')}\n"
            f"Policy CSL        : {row.get('policy_csl')}\n"
            "\n=== ANOMALY SCORE BREAKDOWN ===\n"
            f"Overall Score  : {score:.1f} / 100  (LOWER_THRESHOLD = {AnomalyScorer.LOWER_THRESHOLD})\n"
            f"Priority Tier  : {row.get('priority_tier')}\n"
            f"Rules Triggered: {row.get('triggered_rules', 'None')}\n"
            "\nRule Indicator Values (0=no signal, 1=maximum signal):\n"
            f"  R1 Amount Z-Score       : {r('r1_amount_zscore'):.3f}\n"
            f"  R2 Severity Mismatch    : {r('r2_severity_mismatch'):.3f}\n"
            f"  R3 Injury/Vehicle Ratio : {r('r3_injury_ratio'):.3f}\n"
            f"  R4 Staged Accident      : {r('r4_staged_accident'):.3f}\n"
            f"  R5 Documentation Gap    : {r('r5_doc_gap'):.3f}\n"
            f"  R6 Theft with Injury    : {r('r6_theft_with_injury'):.3f}\n"
            f"  R7 Ghost Injury         : {r('r7_ghost_injury'):.3f}\n"
            f"  R8 Injury Per Person    : {r('r8_injury_per_person'):.3f}\n"
            "\n=== INSTRUCTIONS ===\n"
            "Write a professional investigation brief with these three sections.\n"
            "Use the SPECIFIC DATA POINTS above - do not make generic statements.\n"
            "\n**SECTION 1 - WHY THIS CLAIM IS SUSPICIOUS**\n"
            "Reference actual numbers (dollar amounts, witness count, processing days).\n"
            "\n**SECTION 2 - RISK FACTORS PRESENT**\n"
            "List each triggered rule and explain the fraud risk it represents.\n"
            "\n**SECTION 3 - RECOMMENDED INVESTIGATOR ACTIONS**\n"
            "Provide 3-5 specific, actionable next steps (document to request, party to contact).\n"
            "\nKeep the brief under 400 words. Use professional language suitable for a regulated "
            "environment. Do NOT include analyst names, sign-off lines, or placeholders. "
            "Start directly with Section 1."
        )


#### Run Pipeline — Step 1: Load data and compute raw anomaly scores
Runs enrichment, all 8 rules, and CRITIC weighting to produce `anomaly_score`.
Thresholds are **not applied yet** — the distribution chart in the next cell
determines where the tier boundaries should be set.

In [0]:
# Step 1 — Load and score (raw anomaly_score only, no tiers assigned yet).
# _apply_critic_weights computes anomaly_score but priority_tier is set
# using AnomalyScorer.LOWER_THRESHOLD / HIGH_THRESHOLD class attributes.
# We override those to sentinel values here so tiers are visually meaningless
# until the thresholds are confirmed from the chart in the next cell.

enricher = ClaimsEnricher(schema_silver=SCHEMA_S, schema_gold=SCHEMA_G)
scorer   = AnomalyScorer()
reporter = AnomalyReporter(schema_gold=SCHEMA_G, run_id=RUN_ID, scorer=scorer)

claims, total_claims = enricher.load()

# Run population stats and all 8 rules + CRITIC to produce anomaly_score.
# Temporarily set thresholds to 0/100 so no meaningful tier is assigned
# before the distribution chart confirms the correct boundary values.
AnomalyScorer.LOWER_THRESHOLD = 0
AnomalyScorer.HIGH_THRESHOLD  = 100
claims = scorer.score(claims)

print(f"Raw anomaly scores computed for {total_claims:,} claims.")
print("Inspect the distribution chart below before thresholds are fixed.")


#### Anomaly Score Distribution - threshold justification chart.
##### PURPOSE: 
- Visualise how anomaly_score is distributed across the population before thresholds are applied. The chart is the basis for choosing
- LOWER_THRESHOLD = 40 (MEDIUM) and HIGH_THRESHOLD = 65 (HIGH):
- The bulk of claims cluster in the 0-30 range (normal behaviour).
- A clear thinning of the distribution is visible above 40, marking the point where multi-rule co-occurrence begins (genuine anomalies).
- Claims above 65 form a distinct high-density tail of compound signals.


In [0]:
score_pdf = claims.select("anomaly_score").toPandas()

# Build 10-point buckets: 0-10, 10-20, ..., 90-100
bins   = list(range(0, 101, 10))
labels = [f"{b}-{b+10}" for b in bins[:-1]]
score_pdf["bucket"] = pd.cut(
    score_pdf["anomaly_score"],
    bins          = bins,
    labels        = labels,
    right         = False,    # intervals are [left, right)
    include_lowest= True
)
bucket_counts = (
    score_pdf.groupby("bucket", observed=True)
             .size()
             .reindex(labels, fill_value=0)
             .reset_index()
)
bucket_counts.columns = ["bucket", "count"]

# All bars uniform colour — thresholds have not been decided yet.
# The shape of this distribution informs the threshold choice in the next cell.
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("#0f1117")
ax.set_facecolor("#0f1117")

bars = ax.bar(
    bucket_counts["bucket"],
    bucket_counts["count"],
    color  = "#4e9af1",
    width  = 0.7,
    zorder = 2,
)

# Annotate each bar with its count
for bar, count in zip(bars, bucket_counts["count"]):
    if count > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5,
            str(count),
            ha="center", va="bottom", fontsize=9,
            fontweight="bold", color="white"
        )

# Styling
ax.set_xlabel("Anomaly Score Bucket", color="white", fontsize=11)
ax.set_ylabel("Number of Claims",     color="white", fontsize=11)
ax.set_title(
    "Anomaly Score Distribution — Raw (pre-threshold)\n"
    "Observe where the population thins to determine MEDIUM and HIGH boundaries",
    color="white", fontsize=12, fontweight="bold"
)
ax.tick_params(colors="white")
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, color="#2d3748", linewidth=0.7, zorder=1)
ax.set_axisbelow(True)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
display(fig)
plt.close()

print("Bucket distribution:")
print(bucket_counts.to_string(index=False))

#### Step 2: Fix Thresholds and Finalise Scoring
Based on the distribution chart above:
- The score distribution shows a clear natural break above **40** where multi-rule co-occurrence begins
- Claims above **65** form a distinct compound-signal tail

Set `MEDIUM_THRESHOLD = 40` and `HIGH_THRESHOLD = 65`, then re-apply tiers,
business weights, and triggered rules to finalise the scored DataFrame.

In [0]:
# Step 2 — Fix thresholds based on the distribution chart above, then finalise.
# The distribution shows a clear thinning above 40 (multi-rule co-occurrence begins)
# and a distinct compound-signal tail above 65.
# Update class attributes so all tier assignments use the confirmed values.

AnomalyScorer.LOWER_THRESHOLD = 40   # MEDIUM: score >= 40 and < 65
AnomalyScorer.HIGH_THRESHOLD  = 65   # HIGH:   score >= 65

# Re-run score() with confirmed thresholds — applies tiers, business weights,
# and triggered_rules. anomaly_score itself does not change between runs.
claims = scorer.score(claims)

flagged_count = claims.filter(F.col("priority_tier").isin(["HIGH","MEDIUM"])).count()
claims.groupBy("priority_tier").count().orderBy("priority_tier").show()
print(f"Total: {total_claims:,}  |  Flagged: {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")

# Confirmed distribution chart — now with threshold lines and tier colouring
score_pdf = claims.select("anomaly_score").toPandas()
bins   = list(range(0, 101, 10))
labels = [f"{b}-{b+10}" for b in bins[:-1]]
score_pdf["bucket"] = pd.cut(
    score_pdf["anomaly_score"],
    bins=bins, labels=labels, right=False, include_lowest=True
)
bucket_counts = (
    score_pdf.groupby("bucket", observed=True)
             .size()
             .reindex(labels, fill_value=0)
             .reset_index()
)
bucket_counts.columns = ["bucket", "count"]

def _bar_color(label):
    lo = int(label.split("-")[0])
    if lo >= AnomalyScorer.HIGH_THRESHOLD:  return "#e05252"   # HIGH - red
    if lo >= AnomalyScorer.LOWER_THRESHOLD: return "#f0a500"   # MEDIUM - amber
    return "#4a5568"                                            # LOW - grey

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("#0f1117")
ax.set_facecolor("#0f1117")

bars = ax.bar(
    bucket_counts["bucket"],
    bucket_counts["count"],
    color  = [_bar_color(l) for l in bucket_counts["bucket"]],
    width  = 0.7,
    zorder = 2,
)
for bar, count in zip(bars, bucket_counts["count"]):
    if count > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.5,
            str(count),
            ha="center", va="bottom", fontsize=9,
            fontweight="bold", color="white"
        )

x_labels = list(bucket_counts["bucket"])

def _threshold_x(x_labels, threshold):
    for idx, label in enumerate(x_labels):
        lo, hi = int(label.split("-")[0]), int(label.split("-")[1])
        if lo <= threshold < hi:
            return idx - 0.5
    return len(x_labels) - 0.5

lower_x = _threshold_x(x_labels, AnomalyScorer.LOWER_THRESHOLD)
high_x  = _threshold_x(x_labels, AnomalyScorer.HIGH_THRESHOLD)

ax.axvline(lower_x, color="#f0a500", linewidth=2, linestyle="--", zorder=3)
ax.axvline(high_x,  color="#e05252", linewidth=2, linestyle="--", zorder=3)

ymax = bucket_counts["count"].max()
ax.text(lower_x + 0.08, ymax * 0.92,
        f"MEDIUM threshold\n(score >= {AnomalyScorer.LOWER_THRESHOLD})",
        color="#f0a500", fontsize=9, fontweight="bold")
ax.text(high_x  + 0.08, ymax * 0.75,
        f"HIGH threshold\n(score >= {AnomalyScorer.HIGH_THRESHOLD})",
        color="#e05252", fontsize=9, fontweight="bold")

from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor="#4a5568", label="LOW  (score < 40)"),
        Patch(facecolor="#f0a500", label="MEDIUM  (40 <= score < 65)"),
        Patch(facecolor="#e05252", label="HIGH  (score >= 65)"),
    ],
    facecolor="#1e2235", labelcolor="white", fontsize=9, loc="upper right"
)
ax.set_xlabel("Anomaly Score Bucket", color="white", fontsize=11)
ax.set_ylabel("Number of Claims",     color="white", fontsize=11)
ax.set_title(
    "Anomaly Score Distribution — Confirmed Thresholds\n"
    "Bulk of claims cluster 0-30 (normal); flagged claims concentrate above 40",
    color="white", fontsize=12, fontweight="bold"
)
ax.tick_params(colors="white")
ax.spines[["top","right","left","bottom"]].set_visible(False)
ax.yaxis.grid(True, color="#2d3748", linewidth=0.7, zorder=1)
ax.set_axisbelow(True)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
display(fig)
plt.close()

# Percentile confirmation — numerically documents the threshold choice
total       = len(score_pdf)
below_lower = (score_pdf["anomaly_score"] < AnomalyScorer.LOWER_THRESHOLD).sum()
below_high  = (score_pdf["anomaly_score"] < AnomalyScorer.HIGH_THRESHOLD).sum()
print(f"Claims below LOWER_THRESHOLD ({AnomalyScorer.LOWER_THRESHOLD}) : "
      f"{below_lower:,} / {total:,}  ({100*below_lower/total:.1f}th percentile)")
print(f"Claims below HIGH_THRESHOLD  ({AnomalyScorer.HIGH_THRESHOLD}) : "
      f"{below_high:,} / {total:,}  ({100*below_high/total:.1f}th percentile)")

#### CRITIC vs Business Weights - coverage comparison

In [0]:

# CRITIC vs Business Weights - coverage comparison chart.

reporter.plot_comparison(claims)


###**Why CRITIC is the Correct Starting Point for PrimeInsurance** <br>###
The question is not 'which method is right?' but 'which method should anchor the weights, and how should the other inform it?' <br>
Our recommendation is CRITIC as the primary method, with business rule constraints applied as a floor.

**Reason 1: Auditability Under Regulatory Pressure** <br>
PrimeInsurance is operating under a 90-day regulatory deadline with auditors already questioning data integrity. A weight assigned as '25 because it feels like a critical error' will not survive an external audit. A weight derived from Pearson correlation and standard deviation on the actual claims population will. CRITIC produces a fully reproducible number — run it again on next quarter's data and the weights update automatically.

**Reason 2: The R1/R2 Collinearity Problem is Real** <br>
The 0.999 correlation between R1 (High Amount) and R2 (Severity Mismatch) is not a theoretical concern — it is a structural feature of how these rules were engineered. Both use total_claim_amount as a primary input. Without CRITIC's collinearity penalty, the original weights (25+20=45 pts) effectively count the financial anomaly dimension twice. CRITIC correctly identifies and corrects this. No business rule approach catches this without explicit inter-rule correlation analysis.

**Reason 3: The Dataset is Large Enough for Statistical Methods** <br>
The PrimeInsurance silver layer contains ~3,000 claims. With this sample size, standard deviations and correlations are stable and reliable. The statistical mass justifies CRITIC's assumptions.

**Reason 4: Business Rules Still Have a Role — As a Constraint** <br>
The CRITIC-derived weights for R4 (27.17%) and R5 (23.62%) reflect their statistical independence, not their predictive power. A pure CRITIC implementation would over-weight these weak predictors. The correct approach is to use CRITIC weights as the baseline but apply a minimum floor informed by business rules — ensuring that R1 and R2 (the strongest fraud loss predictors) cannot be reduced below a threshold that domain experts consider operationally meaningful.

**Recommended Hybrid Approach** <br>
1. Run CRITIC to derive statistically grounded baseline weights
2. Check each weight against a business rule minimum floor (e.g., R1 and R2 must each be at least 20% given their proven correlation with claim amount)
3. Document where the floor overrides CRITIC and why — this becomes the audit trail
4. Re-run CRITIC quarterly as new claims data accumulates to detect weight drift

#### AI investigation briefs for all flagged claims

In [0]:

briefs = reporter.generate_briefs(claims)
reporter.write_outputs(claims, briefs, total_claims)


In [0]:
# COMMAND ----------
# Verification - schema checks and sample output.

print("=== SCHEMA: claim_anomaly_scores ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_scores").printSchema()

print("=== SCHEMA: claim_anomaly_explanations ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_explanations").printSchema()

print("=== PRIORITY TIER DISTRIBUTION ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_scores")
    .groupBy("priority_tier")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.avg("anomaly_score"), 2).alias("avg_score"),
        F.round(F.avg("total_claim_amount"), 2).alias("avg_claim_amount"),
    )
    .orderBy(F.desc("avg_score"))
    .show()
)

print("=== TOP 5 HIGH PRIORITY CLAIMS ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
    .filter(F.col("priority_tier") == "HIGH")
    .orderBy(F.desc("anomaly_score"))
    .select("claimid", "anomaly_score", "triggered_rules",
            "total_claim_amount", "incident_severity",
            "number_of_vehicles_involved", "witnesses")
    .limit(5)
    .show(truncate=False)
)

print("=== SAMPLE AI INVESTIGATION BRIEFS (top 3) ===\n")
for row in (spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
        .filter((F.col("priority_tier") == "HIGH") & F.col("ai_investigation_brief").isNotNull())
        .orderBy(F.desc("anomaly_score"))
        .select("claimid","anomaly_score","priority_tier","triggered_rules","ai_investigation_brief")
        .limit(3).collect()):
    print("=" * 80)
    print(f"CLAIM ID    : {row['claimid']}")
    print(f"SCORE       : {row['anomaly_score']} / 100")
    print(f"TIER        : {row['priority_tier']}")
    print(f"RULES FIRED : {row['triggered_rules']}")
    print("-" * 80)
    print(row["ai_investigation_brief"])
    print()


#### Run summary

In [0]:
# COMMAND ----------
# Run summary.

print("\n" + "=" * 60)
print("CLAIMS ANOMALY ENGINE - RUN COMPLETE")
print(f"  Model            : {MODEL_ID}")
print(f"  Run ID           : {RUN_ID}")
print(f"  Total claims     : {total_claims:,}")
print(f"  Flagged claims   : {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")
print(f"  AI briefs written: {len(briefs)}  (all flagged claims)")
print(f"  LOWER_THRESHOLD  : {AnomalyScorer.LOWER_THRESHOLD}")
print(f"  HIGH_THRESHOLD   : {AnomalyScorer.HIGH_THRESHOLD}")
print(f"  Scores table     : {SCHEMA_G}.claim_anomaly_scores")
print(f"  Briefs table     : {SCHEMA_G}.claim_anomaly_explanations")
print(f"  Weight audit     : {SCHEMA_G}.claim_anomaly_weights")
print("=" * 60)
